# Compare experiments

Walks `experiments/` and collects every recorded evaluation into a
DataFrame. Use it to compare runs across different pretraining setups,
different eval parameters, or both.

In [ ]:
import os, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "coffe" / "runners" / "experiments.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Could not locate repo root containing coffe/runners/experiments.py")
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

In [ ]:
import pandas as pd
from coffe.runners.experiments import ExperimentLogger, load_all_evaluations

logger = ExperimentLogger()
experiments = logger.list_experiments()
print(f"Found {len(experiments)} experiment(s).")
pd.DataFrame(experiments)[["name", "description", "status", "started_at", "finished_at"]]

In [ ]:
rows = load_all_evaluations()
df = pd.DataFrame(rows)
print(f"{len(df)} evaluation runs across all experiments.")
df

In [ ]:
# Compact comparison view.
if not df.empty:
    cols = ["experiment", "eval_name", "dataset", "n_way", "k_shot",
            "distance_metric", "temperature", "prototype_mode",
            "OA_mean", "OA_ci", "AA_mean", "Kappa_mean"]
    display(df[cols].sort_values(["dataset", "n_way", "k_shot", "OA_mean"], ascending=[True, True, True, False]))

In [ ]:
# Per-(dataset, n_way, k_shot) leaderboard.
if not df.empty:
    leaderboard = (
        df.sort_values("OA_mean", ascending=False)
          .groupby(["dataset", "n_way", "k_shot"])
          .head(5)[["dataset", "n_way", "k_shot", "experiment", "eval_name",
                    "distance_metric", "temperature", "OA_mean", "OA_ci"]]
    )
    display(leaderboard)

In [ ]:
# OA across runs, grouped by dataset.
if not df.empty:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    df_sorted = df.sort_values(["dataset", "OA_mean"], ascending=[True, False])
    labels = df_sorted["experiment"] + "/" + df_sorted["eval_name"]
    ax.errorbar(range(len(df_sorted)), df_sorted["OA_mean"], yerr=df_sorted["OA_ci"],
                fmt="o", capsize=3)
    ax.set_xticks(range(len(df_sorted)))
    ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
    ax.set_ylabel("OA (%) \u00b1 95% CI")
    ax.set_title("Overall accuracy across evaluation runs")
    plt.tight_layout()
    plt.show()